# Chapter 7: Matrix Examples

Python companion notes for *Introduction to Applied Linear Algebra* (Boyd & Vandenberghe). Explanations are written to accompany the code; for the full text see the original book.

## 7.1 Geometric transformations

A $2\times 2$ rotation matrix turns points about the origin. Multiplying each point by it rotates the whole set — here by $\pi/3$ (60°).

In [ ]:
import numpy as np
Rot = lambda theta: [[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta), np.cos(theta)]]
R = Rot(np.pi / 3)
R

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
points = np.array([[1, 0], [1.5, 0], [2, 0], [1, 0.25], [1.5, 0.25], [1, 0.5]])
rpoints = np.array([R @ p for p in points])  # rotate each point
plt.scatter([c[0] for c in points], [c[1] for c in points])
plt.scatter([c[0] for c in rpoints], [c[1] for c in rpoints])
plt.show()

## 7.2 Selectors

### Reverser matrix

Take the identity and flip its rows: multiplying by the result reverses a vector's order.

In [ ]:
import numpy as np
reverser = lambda n: np.flip(np.eye(n), axis=0)
A = reverser(5)
A

### Permutation matrix

A permutation matrix reorders entries. Multiplying by `A` here rearranges `x`; the same effect is available directly with fancy indexing `x[[2, 0, 1]]`.

In [ ]:
import numpy as np
A = np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]])
x = np.array([0.2, -1.7, 2.4])
A @ x

In [ ]:
x[[2, 0, 1]]

## 7.3 Incidence matrix

The incidence matrix describes a directed graph: each column is an edge, with $-1$ at its start node and $+1$ at its end. A **circulation** is a flow that balances at every node, so `A @ x` comes out all zeros.

In [ ]:
import numpy as np
A = np.array([[-1, -1, 0, 1, 0], [1, 0, -1, 0, 0], [0, 0, 1, -1, -1], [0, 1, 0, 0, 1]])
xcirc = np.array([1, -1, 1, 0, 1])  # a circulation
A @ xcirc

Adding a source vector `s`, the quantity `A @ x + s` gives the net flow arriving at each node.

In [ ]:
import numpy as np
s = np.array([1, 0, -1, 0])
x = np.array([0.6, 0.3, 0.6, -0.1, -0.3])
A @ x + s

### Dirichlet energy

$\|A^T v\|^2$ measures how much a signal `v` varies across the graph's edges — small for a smooth signal, large for a rough one.

In [ ]:
import numpy as np
A = np.array([[-1, -1, 0, 1, 0], [1, 0, -1, 0, 0], [0, 0, 1, -1, -1], [0, 1, 0, 0, 1]])
vsmooth = np.array([1, 2, 2, 1])
np.linalg.norm(A.T @ vsmooth)**2   # smooth: small energy

In [ ]:
vrough = np.array([1, -1, 2, -1])
np.linalg.norm(A.T @ vrough)**2    # rough: large energy

## 7.4 Convolution

Convolving two coefficient vectors multiplies the corresponding polynomials. `np.convolve` does it directly.

$$p(x) = (1+x)(2-x+x^2)(1+x-2x^2) = 2 + 3x - 3x^2 - x^3 + x^4 - 2x^5$$

In [ ]:
import numpy as np
a = np.array([1, 1])       # 1 + x
b = np.array([2, -1, 1])   # 2 - x + x^2
c = np.array([1, 1, -2])   # 1 + x - 2x^2
d = np.convolve(np.convolve(a, b), c)
d

### Toeplitz matrix

Convolution can also be written as multiplication by a **Toeplitz** matrix (each column is the vector `b` shifted down by one). Building it confirms it matches `np.convolve`.

In [ ]:
import numpy as np
b = np.array([-1, 2, 3])
a = np.array([-2, 3, -1, 1])

def toeplitz(b, n):
    m = len(b)
    T = np.zeros((n + m - 1, n))
    for j in range(n):
        T[j:j+m, j] = b
    return T

Tb = toeplitz(b, len(a))
Tb

In [ ]:
Tb @ a, np.convolve(b, a)  # same result

The built-in `np.convolve` is far faster than forming the matrix and multiplying — a nice illustration that the right algorithm beats a naive one.

In [ ]:
import numpy as np
import time
m = n = 2000
b = np.random.normal(size=n)
a = np.random.normal(size=m)
start = time.time()
ctoep = toeplitz(b, n) @ a
end = time.time()
print(end - start)

In [ ]:
start = time.time()
cconv = np.convolve(a, b)
end = time.time()
print(end - start)

In [ ]:
np.linalg.norm(ctoep - cconv)  # essentially zero